# 01 Data Cleaning and Preparation

## Dataset Overview

This dataset contains news articles related to AI, collected from web and contains 200K records. Each record has the following fields:
- `url`: the URL of the news article;
- `date`: the publication date of the news article;
- `language`: just 'en' for 'English';
- `title`: the title of the news article;
- `text`: the main content of the news article;

There are some facts about processing given dataset:
- Text is web-crawled and contains **large amount of noises** like boiler plates and off-topic content (like Ads);
- Trivial data cleaning methods like pattern matching might help, but is far from enough to eliminate off-topic content;
- News titles often contain analogies and idioms, so useful information may be lost if we only use titles;
- Simple truncation (like 512 token length) does not work on the scale of this dataset;

So we decided to LM that supports text-to-text tasks to clean the text. Instruction models like T5 and FLAN-T5 are good candidates, but a more powerful model like GPT or Gemini is more efficient. Main advantages are:
- Better understanding of context and ability to filter out noises;
- Hosted API is more efficient and resource-saving than local deployment of large models;

## Pipeline Overview

- **Basic cleaning**: remove empty rows and duplicates based on titles;
- **LLM instruction**: summarize the content into a concise summary that captures the main topic and key information;

## Downstream tasks
- NER: industry/company extraction;
- Impact direction analysis, mechanism tagging (automation/augmentation);
- Topic discovery.

In [18]:
import os
import polars as pl
import tqdm.notebook as tqdm

In [19]:
if os.path.exists('news_final_project.parquet'):
    df_news_final_project = pl.read_parquet('news_final_project.parquet')
else:
    df_news_final_project = pl.read_parquet('https://storage.googleapis.com/msca-bdp-data-open/news_final_project/news_final_project.parquet')
    df_news_final_project.write_parquet('news_final_project.parquet')
df_news_final_project.shape

(199989, 5)

In [20]:
df_news_final_project.head()

url,date,language,title,text
str,date,str,str,str
"""https://blockworks.co/price/ba…",2025-06-23,"""en""","""Bad Idea AI Price (BAD), Marke…","""Bad Idea AI Price (BAD), Marke…"
"""https://boingboing.net/2024/07…",2024-07-01,"""en""","""This AI video of gymnastics mi…",""" This AI video of gymnastics …"
"""https://boingboing.net/2024/09…",2024-09-22,"""en""","""If using AI feels like a chore…",""" If using AI feels like a cho…"
"""https://citylife.capetown/gl/u…",2023-11-10,"""en""","""The Road Ahead: How China's AI…",""" The Road Ahead: How China's…"
"""https://citylife.capetown/kk/u…",2023-11-19,"""en""","""Microsoft and Nvidia to Empowe…",""" Microsoft and Nvidia to Emp…"


## 1.1 Basic Dataset Check

In [21]:
# Strip and check for empty titles and texts

df_news_final_project = df_news_final_project.with_columns(
    [
        pl.col("title").str.strip_chars().alias("title"),
        pl.col("text").str.strip_chars().alias("text"),
    ]
)

title_empty_count = df_news_final_project.select(
    (pl.col("title").str.len_chars() == 0).sum().alias("title_empty_count")
).item()
text_empty_count = df_news_final_project.select(
    (pl.col("text").str.len_chars() == 0).sum().alias("text_empty_count")
).item()

print(f"Number of empty titles: {title_empty_count}")
print(f"Number of empty texts: {text_empty_count}")

Number of empty titles: 0
Number of empty texts: 0


In [22]:
# Remove duplicates based on titles

original_count = df_news_final_project.height

df_news_final_project = df_news_final_project.unique(subset="title", keep="first", maintain_order=True)
deduped_count = df_news_final_project.height

print(f"Original count: {original_count}")
print(f"Deduped count: {deduped_count}, removed {original_count - deduped_count} duplicates based on title")

Original count: 199989
Deduped count: 164151, removed 35838 duplicates based on title


## 1.2 Cleaning with Pattern Matching

Remove common web-crawl formatting noise while preserving semantic article content.
- Strip HTML tags (`<...>`).
- Remove inline URLs.
- Remove common boilerplate phrases (`subscribe`, `advertisement`, `read more`, etc.).
- Remove obvious JS/CSS fragments.
- Normalize repeated whitespace and extra line breaks.

In [23]:
# Remove HTML / web crawl artifacts with Polars regex replacements

remove_patterns = [
    r"\bsubscribe\s+now\b",
    r"\badvertisement\b",
    r"\bclick\s+here\b",
    r"\bread\s+more\b",
    r"\bsign\s+up\b",
    r"\bnewsletter\b",
    r"\bcookie\s+policy\b",
    r"\ball\s+rights\s+reserved\b",
    r"©",
    r"<[^>]+>",  # html tags
    r"https?://\S+|www\.\S+",  # URLs
    r"(function\s*\(|var\s+\w+\s*=|let\s+\w+\s*=|const\s+\w+\s*=|\{\s*\}|;\s*$|\.css\b|\.js\b)",  # JS/CSS patterns
    r"\b\w{30,}\b",     # Extremely long words (30+ chars) likely to be junk
]

def clean_expr(expr: pl.Expr, patterns: list[str]) -> pl.Expr:
    # Apply each pattern replacement, then normalize repeated whitespace.
    for pattern in patterns:
        expr = expr.str.replace_all(pattern, " ")
    return expr.str.replace_all(r"\s\s+", " ").str.strip_chars()

articles_df = df_news_final_project.with_columns(
    [
        clean_expr(pl.col("title"), remove_patterns).alias("title"),
        clean_expr(pl.col("text"), remove_patterns).alias("text"),
    ]
).filter(pl.col("text").str.len_chars() > 0)

raw_title_len = df_news_final_project.select(pl.col("title").str.len_chars().sum()).item()
clean_title_len = articles_df.select(pl.col("title").str.len_chars().sum()).item()
raw_text_len = df_news_final_project.select(pl.col("text").str.len_chars().sum()).item()
clean_text_len = articles_df.select(pl.col("text").str.len_chars().sum()).item()

articles_df = articles_df.select([
    pl.col("date"),
    pl.col('title'),
    pl.col('text')
])

print(f"Title character count: {raw_title_len} -> {clean_title_len}"
      f"({100 * clean_title_len / raw_title_len:.2f}%)")

print(f"Text character count: {raw_text_len} -> {clean_text_len}"
      f"({100 * clean_text_len / raw_text_len:.2f}%)")

Title character count: 13966486 -> 13935984(99.78%)
Text character count: 1551038908 -> 1405061747(90.59%)


In [24]:
# Checkpoint (Row count 164151)
articles_df.write_parquet('010_preprocess.parquet')

## 1.3 Summarize article Using Generative AI (Gemini)

In [25]:
import json
import time

from concurrent.futures import ThreadPoolExecutor
from google import genai
from pydantic import BaseModel


articles_df = pl.read_parquet('010_preprocess.parquet')


if not os.path.exists('genai_api_key.txt'):
    API_KEY = ''
else:
    with open('genai_api_key.txt', 'r') as f:
        API_KEY = f.read().strip()


# Flash lite version of Gemini is efficient and cost effective
instruct_model = 'gemini-2.5-flash-lite'
instruction = 'Summarize the following web news/article content in 1-2 sentences:\n\n'
ai_client = genai.Client(api_key=API_KEY)


class TextSummary(BaseModel):
    summary: str


resp_config = genai.types.GenerateContentConfig(
    thinking_config=genai.types.ThinkingConfig(
        thinking_budget=0,
    ),
    response_mime_type="application/json",
    response_schema=TextSummary.model_json_schema()
)


def summarize_article(text: str) -> TextSummary:
    response = ai_client.models.generate_content(
        model=instruct_model,
        contents=instruction + text,
        config=resp_config
    )
    return TextSummary.model_validate_json(response.text)


def summarize_article_non_json(text: str) -> TextSummary:
    response = ai_client.models.generate_content(
        model=instruct_model,
        contents=instruction + text,
        config=resp_config
    )
    return response.text


def safe_summarize_articles(text: str) -> dict:
    for _ in range(5):  # Retry up to 5 times
        try:
            resp = summarize_article(text)
            return {"error": '', "summary": resp.summary}
        except Exception as e:
            last_exception = e
            time.sleep(.2)  # Backoff before retrying
    return {"error": str(last_exception), "summary": None}

Since 100K calls of GenAI is cost sensitive, we run calls in parallel in batch and save responses after finishing each batch. Then load them and merge into original dataset

In [26]:
resp_cache_dir = 'sums'
os.makedirs(resp_cache_dir, exist_ok=True)

def summarize_and_save_batch(texts: list[str], idx: int) -> list[dict]:
    with ThreadPoolExecutor(max_workers=48) as executor:
        results = list(executor.map(safe_summarize_articles, texts))
    with open(os.path.join(resp_cache_dir, f'batch_{str(idx).zfill(3)}.jsonl'), 'w') as f:
        for res in results:
            f.write(json.dumps(res) + '\n')

In [27]:
texts = articles_df.get_column("text").to_list()

# Uncomment the following to reproduce the summarization and caching process.
# This is commented out to avoid unnecessary API calls during development/testing.

# BATCH_SIZE = 1000
# for i in tqdm.tqdm(list(range(0, len(texts), BATCH_SIZE))):
#     summarize_and_save_batch(texts[i:i+BATCH_SIZE], i // BATCH_SIZE)

In [28]:
# Load from cache

# Uncomment the following to reproduce the summarization and caching process.
# This is commented out because downstream env does not require cache files.

# cache_files = sorted(list(os.listdir(resp_cache_dir)))
# summary_responses = []
# error_count = 0

# for file in cache_files:
#     with open(os.path.join(resp_cache_dir, file), 'r') as f:
#         for line in f:
#             res = json.loads(line)
#             if res["error"]:
#                 error_count += 1
#             summary_responses.append(res)

# print(f"Total summaries: {len(summary_responses)}, Errors: {error_count}")

After retrying, most errors are validation errors, where AI API returns None for response. This is mainly due to dirty data (like crawler blocked by websites).

You can try to explore `texts[3272]` to see an example. Since there are 60 errors (~0.04%), we can safely discard them.

In [29]:
# Uncomment the following to reproduce the summarization and caching process.
# This is commented out because downstream env does not require cache files.

# titles = articles_df.get_column("title").to_list()
# dates = articles_df.get_column("date").to_list()

# res_list = []
# for dt, title, res in zip(dates, titles, summary_responses):
#     if res['error']:
#         continue
#     res_list.append({
#         "date": dt,
#         "title": title,
#         "summary": res["summary"],
#     })
# summary_df = pl.DataFrame(res_list)
# print('Remaining records:', summary_df.height)

# summary_df.write_parquet('011_summary.parquet')